# Детектирование опухолей мозга на МРТ — обучение (Colab)

Runtime → Run all. GPU (T4). Код проекта создаётся автоматически, датасет качается по ключу Roboflow.


## 1. GPU


In [ ]:
!nvidia-smi


## 2. Установка


In [ ]:
!pip install -q ultralytics effdet==0.4.1 'timm>=0.9.12' 'transformers>=4.40' pycocotools roboflow


## 3. Файлы проекта


In [ ]:
import os
os.chdir('/content')
!rm -rf medical_cv_project && mkdir -p medical_cv_project
os.chdir('/content/medical_cv_project')
!mkdir -p configs scripts src/dataset src/models src/training src/evaluation src/utils data/raw data/processed results/plots results/logs
for d in ['src','src/dataset','src/models','src/training','src/evaluation','src/utils']: open(d+'/__init__.py','w').close()


In [ ]:
%%writefile configs/default.yaml
# =====================================================================
# Конфигурация проекта: детектирование опухолей головного мозга на МРТ
# =====================================================================

project:
  name: brain_tumor_detection
  seed: 42                       # глобальный seed для воспроизводимости

# ---------------------------------------------------------------------
# Данные. Структура после prepare_data.py:
#   data/processed/
#     ├── data.yaml              # YOLO-конфиг (path, train, val, test, names)
#     ├── train/{images,labels}
#     ├── val/{images,labels}
#     └── test/{images,labels}
# ---------------------------------------------------------------------
data:
  processed_dir: data/processed
  yaml: data/processed/data.yaml
  # 4 класса (Brain_Tumor_Detect, Roboflow). Порядок — алфавитный, как в data.yaml
  # после prepare_data.py. Если в вашем экспорте классы иные — поправьте здесь.
  class_names: [Glioma_Tumor, Meningioma_Tumor, No_Tumor, Pituitary_Tumor]
  num_classes: 4

# ---------------------------------------------------------------------
# Общие настройки обучения (могут переопределяться на уровне модели)
# ---------------------------------------------------------------------
train:
  epochs: 50
  early_stop_patience: 10        # эпох без улучшения mAP@0.5 на val
  num_workers: 2
  device: cuda                   # cuda | mps | cpu

# Параметры augmentation для torchvision/DETR/EfficientDet пайплайна
augment:
  hflip_p: 0.5
  scale_jitter: 0.10
  brightness_contrast: 0.20
  translate: 0.05

# Нормализация (ImageNet)
normalize:
  mean: [0.485, 0.456, 0.406]
  std:  [0.229, 0.224, 0.225]

# ---------------------------------------------------------------------
# Гиперпараметры по моделям
# ---------------------------------------------------------------------
models:
  yolov8:
    weights: yolov8n.pt
    img_size: 640
    batch: 16
    lr0: 0.01
    optimizer: SGD
  yolov5:
    weights: yolov5su.pt         # YOLOv5s в формате ultralytics
    img_size: 640
    batch: 16
    lr0: 0.01
    optimizer: SGD
  faster_rcnn:
    img_size: 640
    batch: 4
    lr: 0.005
    optimizer: SGD
    step_size: 15
    gamma: 0.1
  ssd:
    img_size: 320
    batch: 8
    lr: 0.01
    optimizer: SGD
  efficientdet:
    variant: tf_efficientdet_d0
    img_size: 512
    batch: 8
    lr: 0.0001
    optimizer: AdamW
  detr:
    weights: facebook/detr-resnet-50
    img_size: 800
    batch: 4
    lr: 0.0001
    optimizer: AdamW

# ---------------------------------------------------------------------
# Оценка
# ---------------------------------------------------------------------
eval:
  iou_thr: 0.5                   # порог для mAP@0.5 / P / R / F1
  score_thr: 0.05                # минимальный confidence для предсказания
  results_csv: results/logs/metrics.csv


In [ ]:
%%writefile src/utils/utils.py
from __future__ import annotations

import os
import random
import logging
from pathlib import Path

import numpy as np
import torch

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device(prefer: str = "cuda") -> torch.device:
    if prefer == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    if prefer == "mps" and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def get_logger(name: str, log_dir: str = "results/logs") -> logging.Logger:
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S")

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    fh = logging.FileHandler(Path(log_dir) / f"{name}.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    return logger

def load_config(path: str = "configs/default.yaml") -> dict:
    import yaml
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def append_metrics(csv_path: str, row: dict) -> None:
    import pandas as pd
    Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
    df_row = pd.DataFrame([row])
    if Path(csv_path).exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, index=False)

def draw_predictions(image, boxes, labels, scores, class_names, out_path, score_thr=0.3):
    import cv2
    if isinstance(image, (str, Path)):
        image = cv2.cvtColor(cv2.imread(str(image)), cv2.COLOR_BGR2RGB)
    img = image.copy()
    palette = [(255, 56, 56), (56, 255, 56), (56, 56, 255),
               (255, 200, 0), (200, 0, 255), (0, 200, 200)]
    for box, lab, sc in zip(boxes, labels, scores):
        if sc < score_thr:
            continue
        x1, y1, x2, y2 = [int(v) for v in box]
        color = palette[int(lab) % len(palette)]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        name = class_names[int(lab)] if int(lab) < len(class_names) else str(int(lab))
        cv2.putText(img, f"{name} {sc:.2f}", (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    return img


In [ ]:
%%writefile src/dataset/dataset.py
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def _list_images(images_dir: Path):
    return sorted([p for p in images_dir.iterdir() if p.suffix.lower() in IMG_EXTS])

def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return x1, y1, x2, y2

class ObjectDetectionDataset(Dataset):

    def __init__(self, split_dir, img_size=640, augment=False, aug_cfg=None,
                 normalize=None):
        self.split_dir = Path(split_dir)
        self.images_dir = self.split_dir / "images"
        self.labels_dir = self.split_dir / "labels"
        self.img_size = img_size
        self.augment = augment
        self.aug = aug_cfg or {}
        self.normalize = normalize
        self.images = _list_images(self.images_dir)
        if not self.images:
            raise FileNotFoundError(f"Не найдено изображений в {self.images_dir}")

    def __len__(self):
        return len(self.images)

    def _read_label(self, img_path: Path):
        label_path = self.labels_dir / (img_path.stem + ".txt")
        boxes, labels = [], []
        if label_path.exists():
            for line in label_path.read_text().strip().splitlines():
                if not line.strip():
                    continue
                cls, cx, cy, w, h = map(float, line.split()[:5])
                boxes.append([cx, cy, w, h])
                labels.append(int(cls))
        return np.array(boxes, dtype=np.float32).reshape(-1, 4), \
               np.array(labels, dtype=np.int64)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]

        boxes_n, labels = self._read_label(img_path)

        img = cv2.resize(img, (self.img_size, self.img_size))

        if self.augment and boxes_n.shape[0] > 0:
            if np.random.rand() < self.aug.get("hflip_p", 0.0):
                img = img[:, ::-1, :].copy()
                boxes_n[:, 0] = 1.0 - boxes_n[:, 0]
            bc = self.aug.get("brightness_contrast", 0.0)
            if bc > 0:
                alpha = 1.0 + np.random.uniform(-bc, bc)
                beta = np.random.uniform(-bc, bc) * 255
                img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        if boxes_n.shape[0] > 0:
            xyxy = np.zeros_like(boxes_n)
            for i, (cx, cy, w, h) in enumerate(boxes_n):
                xyxy[i] = yolo_to_xyxy(cx, cy, w, h, self.img_size, self.img_size)
            xyxy = np.clip(xyxy, 0, self.img_size)
        else:
            xyxy = np.zeros((0, 4), dtype=np.float32)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        if self.normalize:
            mean = torch.tensor(self.normalize["mean"]).view(3, 1, 1)
            std = torch.tensor(self.normalize["std"]).view(3, 1, 1)
            img_t = (img_t - mean) / std

        target = {
            "boxes": torch.as_tensor(xyxy, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "orig_size": torch.tensor([orig_h, orig_w]),
        }
        return img_t, target

def collate_fn(batch):
    images, targets = list(zip(*batch))
    return list(images), list(targets)

def get_dataloaders(processed_dir, img_size=640, batch=8, num_workers=2,
                    augment=True, aug_cfg=None, normalize=None,
                    splits=("train", "val", "test")):
    processed_dir = Path(processed_dir)
    loaders = {}
    for split in splits:
        ds = ObjectDetectionDataset(
            processed_dir / split, img_size=img_size,
            augment=(augment and split == "train"),
            aug_cfg=aug_cfg, normalize=normalize,
        )
        loaders[split] = DataLoader(
            ds, batch_size=batch, shuffle=(split == "train"),
            num_workers=num_workers, collate_fn=collate_fn, pin_memory=True,
        )
    return loaders


In [ ]:
%%writefile src/evaluation/metrics.py
from __future__ import annotations

import numpy as np
import torch

def box_iou(boxes1, boxes2):
    try:
        from torchvision.ops import box_iou as tv_iou
        return tv_iou(boxes1, boxes2)
    except Exception:
        area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
        area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
        lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
        rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
        wh = (rb - lt).clamp(min=0)
        inter = wh[:, :, 0] * wh[:, :, 1]
        union = area1[:, None] + area2[None, :] - inter
        return inter / union.clamp(min=1e-9)

def _ap_from_pr(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

def _match_class(preds, gts, iou_thr):
    n_gt = sum(g.shape[0] for g in gts.values())
    if len(preds) == 0:
        return np.array([]), np.array([]), np.array([]), n_gt

    preds = sorted(preds, key=lambda x: x[0], reverse=True)
    matched = {idx: np.zeros(g.shape[0], dtype=bool) for idx, g in gts.items()}
    tp = np.zeros(len(preds))
    fp = np.zeros(len(preds))
    scores = np.array([p[0] for p in preds])

    for i, (score, img_idx, box) in enumerate(preds):
        g = gts.get(img_idx)
        if g is None or g.shape[0] == 0:
            fp[i] = 1
            continue
        ious = box_iou(box.unsqueeze(0), g).squeeze(0)
        best = torch.argmax(ious).item()
        if ious[best].item() >= iou_thr and not matched[img_idx][best]:
            tp[i] = 1
            matched[img_idx][best] = True
        else:
            fp[i] = 1
    return tp, fp, scores, n_gt

def _prf(tp, fp, n_gt):
    p = tp / max(tp + fp, 1e-9)
    r = tp / max(n_gt, 1e-9)
    f1 = 2 * p * r / max(p + r, 1e-9)
    return p, r, f1

def compute_metrics(predictions, targets, num_classes, iou_thr=0.5,
                    iou_range=None, class_names=None):
    if iou_range is None:
        iou_range = [round(x, 2) for x in np.arange(0.5, 1.0, 0.05)]

    per_cls_preds = {c: [] for c in range(num_classes)}
    per_cls_gts = {c: {} for c in range(num_classes)}

    for img_idx, (pred, tgt) in enumerate(zip(predictions, targets)):
        for c in range(num_classes):
            m = tgt["labels"] == c
            if m.sum() > 0:
                per_cls_gts[c][img_idx] = tgt["boxes"][m]
        for box, score, lab in zip(pred["boxes"], pred["scores"], pred["labels"]):
            per_cls_preds[int(lab)].append((float(score), img_idx, box))

    ap_main, ap_coco = [], []

    cls_arrays = {}
    names = {}

    for c in range(num_classes):
        name = class_names[c] if class_names and c < len(class_names) else f"class_{c}"
        names[c] = name
        tp, fp, scores, n_gt = _match_class(per_cls_preds[c], per_cls_gts[c], iou_thr)
        cls_arrays[c] = (scores, tp, fp, n_gt)

        if len(tp) == 0:
            if n_gt > 0:
                ap_main.append(0.0)
        else:
            order = np.argsort(-scores)
            tp_c, fp_c = np.cumsum(tp[order]), np.cumsum(fp[order])
            recall = tp_c / max(n_gt, 1e-9)
            precision = tp_c / np.maximum(tp_c + fp_c, 1e-9)
            ap_main.append(_ap_from_pr(recall, precision))

        if n_gt > 0:
            aps_range = []
            for thr in iou_range:
                tp_t, fp_t, sc_t, ng = _match_class(per_cls_preds[c], per_cls_gts[c], thr)
                if len(tp_t) == 0:
                    aps_range.append(0.0); continue
                o = np.argsort(-sc_t)
                tpc, fpc = np.cumsum(tp_t[o]), np.cumsum(fp_t[o])
                rec = tpc / max(ng, 1e-9)
                prec = tpc / np.maximum(tpc + fpc, 1e-9)
                aps_range.append(_ap_from_pr(rec, prec))
            ap_coco.append(float(np.mean(aps_range)))

    all_scores = np.concatenate([a[0] for a in cls_arrays.values()]) \
        if any(len(a[0]) for a in cls_arrays.values()) else np.array([])
    total_gt = sum(a[3] for a in cls_arrays.values())

    best = {"f1": -1.0, "p": 0.0, "r": 0.0, "conf": 0.25}
    if all_scores.size > 0:

        cand = np.unique(np.round(all_scores, 3))
        if cand.size > 200:
            cand = np.quantile(all_scores, np.linspace(0.0, 0.99, 200))
        for t in cand:
            tp_sum = fp_sum = 0
            for scores, tp, fp, _ in cls_arrays.values():
                if len(scores) == 0:
                    continue
                m = scores >= t
                tp_sum += int(tp[m].sum()); fp_sum += int(fp[m].sum())
            p, r, f1 = _prf(tp_sum, fp_sum, total_gt)
            if f1 > best["f1"]:
                best = {"f1": f1, "p": p, "r": r, "conf": float(t)}

    per_class = {}
    bt = best["conf"]
    for c in range(num_classes):
        scores, tp, fp, n_gt = cls_arrays[c]
        name = names[c]

        if len(tp):
            order = np.argsort(-scores)
            tpc, fpc = np.cumsum(tp[order]), np.cumsum(fp[order])
            rec = tpc / max(n_gt, 1e-9); prec = tpc / np.maximum(tpc + fpc, 1e-9)
            ap = _ap_from_pr(rec, prec)
        else:
            ap = 0.0
        if len(scores):
            m = scores >= bt
            p, r, f1 = _prf(int(tp[m].sum()), int(fp[m].sum()), n_gt)
        else:
            p = r = f1 = 0.0
        per_class[name] = dict(AP=round(ap, 4), precision=round(p, 4),
                               recall=round(r, 4), f1=round(f1, 4), n_gt=n_gt)

    return {
        "mAP@0.5": round(float(np.mean(ap_main)) if ap_main else 0.0, 4),
        "mAP@0.5:0.95": round(float(np.mean(ap_coco)) if ap_coco else 0.0, 4),
        "precision": round(best["p"], 4),
        "recall": round(best["r"], 4),
        "f1": round(best["f1"], 4),
        "conf_thr": round(best["conf"], 3),
        "per_class": per_class,
    }


In [ ]:
%%writefile src/models/yolo_base.py
from __future__ import annotations

class UltralyticsYOLO:
    name = "yolo"

    def __init__(self, weights, img_size=640, batch=16, lr0=0.01,
                 optimizer="SGD", seed=42, project="results/runs", run_name=None):
        from ultralytics import YOLO
        self.weights = weights
        self.img_size = img_size
        self.batch = batch
        self.lr0 = lr0
        self.optimizer = optimizer
        self.seed = seed
        self.project = project
        self.run_name = run_name or self.name
        self.model = YOLO(weights)

    def train(self, data_yaml, epochs=50, patience=10, device=0):
        self.model.train(
            data=data_yaml, epochs=epochs, imgsz=self.img_size, batch=self.batch,
            lr0=self.lr0, optimizer=self.optimizer, seed=self.seed,
            patience=patience, project=self.project, name=self.run_name,
            device=device, plots=True, exist_ok=True,
        )
        return self.model

    def evaluate(self, data_yaml, split="test", class_names=None):
        res = self.model.val(data=data_yaml, split=split, imgsz=self.img_size,
                             project=self.project, name=f"{self.run_name}_{split}",
                             exist_ok=True)
        box = res.box
        p, r = float(box.mp), float(box.mr)
        f1 = 2 * p * r / max(p + r, 1e-9)
        out = {
            "mAP@0.5": round(float(box.map50), 4),
            "mAP@0.5:0.95": round(float(box.map), 4),
            "precision": round(p, 4),
            "recall": round(r, 4),
            "f1": round(f1, 4),
            "per_class": {},
        }

        if class_names is not None:
            try:
                for i, ap in enumerate(box.ap50):
                    nm = class_names[i] if i < len(class_names) else f"class_{i}"
                    out["per_class"][nm] = {"AP": round(float(ap), 4)}
            except Exception:
                pass
        return out

    def predict(self, image, conf=0.25):
        r = self.model.predict(image, imgsz=self.img_size, conf=conf, verbose=False)[0]
        return {
            "boxes": r.boxes.xyxy.cpu(),
            "scores": r.boxes.conf.cpu(),
            "labels": r.boxes.cls.cpu().long(),
        }


In [ ]:
%%writefile src/models/yolov8_model.py
from __future__ import annotations

from .yolo_base import UltralyticsYOLO

class YOLOv8Model(UltralyticsYOLO):
    name = "yolov8"

    @classmethod
    def from_config(cls, cfg, seed=42):
        m = cfg["models"]["yolov8"]
        return cls(weights=m["weights"], img_size=m["img_size"], batch=m["batch"],
                   lr0=m["lr0"], optimizer=m["optimizer"], seed=seed, run_name="yolov8")


In [ ]:
%%writefile src/models/yolov5_model.py
from __future__ import annotations

from .yolo_base import UltralyticsYOLO

class YOLOv5Model(UltralyticsYOLO):
    name = "yolov5"

    @classmethod
    def from_config(cls, cfg, seed=42):
        m = cfg["models"]["yolov5"]
        return cls(weights=m["weights"], img_size=m["img_size"], batch=m["batch"],
                   lr0=m["lr0"], optimizer=m["optimizer"], seed=seed, run_name="yolov5")


In [ ]:
%%writefile src/models/faster_rcnn.py
from __future__ import annotations

import torch

class FasterRCNNModel:
    name = "faster_rcnn"
    needs_normalize = False

    def __init__(self, num_classes, img_size=640, lr=0.005,
                 step_size=15, gamma=0.1, optimizer="SGD"):
        import torchvision
        from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
        self.num_classes = num_classes
        self.img_size = img_size
        self.lr = lr
        self.step_size = step_size
        self.gamma = gamma
        self.optimizer_name = optimizer

        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
            weights="DEFAULT", min_size=img_size, max_size=img_size)
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features

        self.model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
        self.device = torch.device("cpu")

    def to(self, device):
        self.device = device
        self.model.to(device)
        return self

    def train_mode(self):
        self.model.train()

    def eval_mode(self):
        self.model.eval()

    def parameters(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def build_optimizer(self):
        if self.optimizer_name.upper() == "SGD":
            opt = torch.optim.SGD(self.parameters(), lr=self.lr,
                                  momentum=0.9, weight_decay=5e-4)
        else:
            opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=self.step_size,
                                                gamma=self.gamma)
        return opt, sched

    def _to_targets(self, targets):
        out = []
        for t in targets:
            out.append({
                "boxes": t["boxes"].to(self.device),
                "labels": (t["labels"] + 1).to(self.device),
            })
        return out

    def train_forward(self, images, targets):
        images = [img.to(self.device) for img in images]

        valid = [(im, t) for im, t in zip(images, targets) if t["boxes"].shape[0] > 0]
        if not valid:
            return torch.zeros(1, requires_grad=True, device=self.device).sum()
        images = [v[0] for v in valid]
        targets = self._to_targets([v[1] for v in valid])
        loss_dict = self.model(images, targets)
        return sum(loss for loss in loss_dict.values())

    @torch.no_grad()
    def predict(self, images):
        self.model.eval()
        images = [img.to(self.device) for img in images]
        outputs = self.model(images)
        preds = []
        for o in outputs:
            preds.append({
                "boxes": o["boxes"].cpu(),
                "scores": o["scores"].cpu(),
                "labels": (o["labels"] - 1).cpu(),
            })
        return preds

    @classmethod
    def from_config(cls, cfg):
        m = cfg["models"]["faster_rcnn"]
        return cls(num_classes=cfg["data"]["num_classes"], img_size=m["img_size"],
                   lr=m["lr"], step_size=m["step_size"], gamma=m["gamma"],
                   optimizer=m["optimizer"])


In [ ]:
%%writefile src/models/ssd.py
from __future__ import annotations

import torch

class SSDModel:
    name = "ssd"
    needs_normalize = False

    def __init__(self, num_classes, img_size=320, lr=0.01, optimizer="SGD"):
        import torchvision
        from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
        from torchvision.models.detection import _utils as det_utils
        from functools import partial

        self.num_classes = num_classes
        self.img_size = img_size
        self.lr = lr
        self.optimizer_name = optimizer

        self.model = torchvision.models.detection.ssdlite320_mobilenet_v3_large(
            weights="DEFAULT")

        in_channels = det_utils.retrieve_out_channels(self.model.backbone,
                                                       (img_size, img_size))
        num_anchors = self.model.anchor_generator.num_anchors_per_location()
        norm_layer = partial(torch.nn.BatchNorm2d, eps=0.001, momentum=0.03)
        self.model.head.classification_head = SSDLiteClassificationHead(
            in_channels, num_anchors, num_classes + 1, norm_layer)

        self.model.transform.min_size = (img_size,)
        self.model.transform.max_size = img_size
        self.device = torch.device("cpu")

    def to(self, device):
        self.device = device
        self.model.to(device)
        return self

    def train_mode(self):
        self.model.train()

    def eval_mode(self):
        self.model.eval()

    def parameters(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def build_optimizer(self):
        if self.optimizer_name.upper() == "SGD":
            opt = torch.optim.SGD(self.parameters(), lr=self.lr,
                                  momentum=0.9, weight_decay=5e-4)
        else:
            opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
        return opt, sched

    def _to_targets(self, targets):
        return [{"boxes": t["boxes"].to(self.device),
                 "labels": (t["labels"] + 1).to(self.device)} for t in targets]

    def train_forward(self, images, targets):
        images = [img.to(self.device) for img in images]
        valid = [(im, t) for im, t in zip(images, targets) if t["boxes"].shape[0] > 0]
        if not valid:
            return torch.zeros(1, requires_grad=True, device=self.device).sum()
        images = [v[0] for v in valid]
        targets = self._to_targets([v[1] for v in valid])
        loss_dict = self.model(images, targets)
        return sum(loss for loss in loss_dict.values())

    @torch.no_grad()
    def predict(self, images):
        self.model.eval()
        images = [img.to(self.device) for img in images]
        outputs = self.model(images)
        return [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(),
                 "labels": (o["labels"] - 1).cpu()} for o in outputs]

    @classmethod
    def from_config(cls, cfg):
        m = cfg["models"]["ssd"]
        return cls(num_classes=cfg["data"]["num_classes"], img_size=m["img_size"],
                   lr=m["lr"], optimizer=m["optimizer"])


In [ ]:
%%writefile src/models/efficientdet.py
from __future__ import annotations

import torch

class EfficientDetModel:
    name = "efficientdet"
    needs_normalize = True

    def __init__(self, num_classes, variant="tf_efficientdet_d0", img_size=512,
                 lr=1e-4, optimizer="AdamW"):
        from effdet import create_model
        self.num_classes = num_classes
        self.img_size = img_size
        self.lr = lr
        self.optimizer_name = optimizer

        self.bench = create_model(
            variant, bench_task="train", num_classes=num_classes,
            pretrained=True, image_size=(img_size, img_size),
            bench_labeler=True,
        )
        self._predict_bench = None
        self.device = torch.device("cpu")

    def to(self, device):
        self.device = device
        self.bench.to(device)
        return self

    def train_mode(self):
        self.bench.train()

    def eval_mode(self):
        self.bench.eval()

    def parameters(self):
        return [p for p in self.bench.parameters() if p.requires_grad]

    def build_optimizer(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
        return opt, sched

    def _build_target(self, targets, batch_size):
        max_n = max((t["boxes"].shape[0] for t in targets), default=1)
        max_n = max(max_n, 1)
        bbox = torch.zeros(batch_size, max_n, 4, device=self.device)
        cls = torch.zeros(batch_size, max_n, device=self.device)
        for i, t in enumerate(targets):
            n = t["boxes"].shape[0]
            if n == 0:
                continue
            b = t["boxes"].to(self.device)

            bbox[i, :n] = b[:, [1, 0, 3, 2]]
            cls[i, :n] = (t["labels"].to(self.device) + 1).float()
        return {
            "bbox": bbox,
            "cls": cls,
            "img_size": torch.tensor([[self.img_size, self.img_size]] * batch_size,
                                     device=self.device).float(),
            "img_scale": torch.ones(batch_size, device=self.device),
        }

    def train_forward(self, images, targets):
        x = torch.stack([img.to(self.device) for img in images])
        target = self._build_target(targets, x.shape[0])
        out = self.bench(x, target)
        return out["loss"]

    @torch.no_grad()
    def predict(self, images):
        from effdet import DetBenchPredict
        if self._predict_bench is None:
            self._predict_bench = DetBenchPredict(self.bench.model).to(self.device)
        self._predict_bench.eval()
        x = torch.stack([img.to(self.device) for img in images])
        det = self._predict_bench(x)
        preds = []
        for d in det:
            d = d.cpu()
            preds.append({
                "boxes": d[:, :4],
                "scores": d[:, 4],
                "labels": (d[:, 5].long() - 1),
            })
        return preds

    @classmethod
    def from_config(cls, cfg):
        m = cfg["models"]["efficientdet"]
        return cls(num_classes=cfg["data"]["num_classes"], variant=m["variant"],
                   img_size=m["img_size"], lr=m["lr"], optimizer=m["optimizer"])


In [ ]:
%%writefile src/models/detr.py
from __future__ import annotations

import torch

class DETRModel:
    name = "detr"
    needs_normalize = True

    def __init__(self, num_classes, weights="facebook/detr-resnet-50",
                 img_size=800, lr=1e-4, optimizer="AdamW"):
        from transformers import DetrForObjectDetection
        self.num_classes = num_classes
        self.img_size = img_size
        self.lr = lr
        self.optimizer_name = optimizer
        self.model = DetrForObjectDetection.from_pretrained(
            weights, num_labels=num_classes, ignore_mismatched_sizes=True)
        self.device = torch.device("cpu")

    def to(self, device):
        self.device = device
        self.model.to(device)
        return self

    def train_mode(self):
        self.model.train()

    def eval_mode(self):
        self.model.eval()

    def parameters(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def build_optimizer(self):

        backbone = [p for n, p in self.model.named_parameters()
                    if "backbone" in n and p.requires_grad]
        rest = [p for n, p in self.model.named_parameters()
                if "backbone" not in n and p.requires_grad]
        opt = torch.optim.AdamW(
            [{"params": rest, "lr": self.lr},
             {"params": backbone, "lr": self.lr * 0.1}], weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
        return opt, sched

    def _build_labels(self, targets):
        labels = []
        for t in targets:
            boxes = t["boxes"].to(self.device)
            if boxes.shape[0] == 0:
                labels.append({
                    "class_labels": torch.zeros(0, dtype=torch.long, device=self.device),
                    "boxes": torch.zeros(0, 4, device=self.device),
                })
                continue

            cx = (boxes[:, 0] + boxes[:, 2]) / 2 / self.img_size
            cy = (boxes[:, 1] + boxes[:, 3]) / 2 / self.img_size
            w = (boxes[:, 2] - boxes[:, 0]) / self.img_size
            h = (boxes[:, 3] - boxes[:, 1]) / self.img_size
            labels.append({
                "class_labels": t["labels"].to(self.device),
                "boxes": torch.stack([cx, cy, w, h], dim=1),
            })
        return labels

    def train_forward(self, images, targets):
        pixel_values = torch.stack([img.to(self.device) for img in images])
        labels = self._build_labels(targets)
        out = self.model(pixel_values=pixel_values, labels=labels)
        return out.loss

    @torch.no_grad()
    def predict(self, images):
        self.model.eval()
        pixel_values = torch.stack([img.to(self.device) for img in images])
        out = self.model(pixel_values=pixel_values)
        logits = out.logits.softmax(-1)[..., :-1]
        boxes = out.pred_boxes
        preds = []
        for i in range(pixel_values.shape[0]):
            scores, labels = logits[i].max(-1)
            b = boxes[i]
            cx, cy, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
            xyxy = torch.stack([
                (cx - w / 2) * self.img_size, (cy - h / 2) * self.img_size,
                (cx + w / 2) * self.img_size, (cy + h / 2) * self.img_size,
            ], dim=1)
            preds.append({"boxes": xyxy.cpu(), "scores": scores.cpu(),
                          "labels": labels.cpu()})
        return preds

    @classmethod
    def from_config(cls, cfg):
        m = cfg["models"]["detr"]
        return cls(num_classes=cfg["data"]["num_classes"], weights=m["weights"],
                   img_size=m["img_size"], lr=m["lr"], optimizer=m["optimizer"])


In [ ]:
%%writefile src/training/train.py
from __future__ import annotations

import time
from pathlib import Path

import torch
from tqdm import tqdm

from src.evaluation.metrics import compute_metrics

@torch.no_grad()
def evaluate_model(model_wrapper, loader, num_classes, iou_thr=0.5,
                   score_thr=0.05, class_names=None):
    model_wrapper.eval_mode()
    all_preds, all_targets = [], []
    for images, targets in tqdm(loader, desc="eval", leave=False):
        preds = model_wrapper.predict(images)
        for p, t in zip(preds, targets):
            keep = p["scores"] >= score_thr
            all_preds.append({"boxes": p["boxes"][keep],
                              "scores": p["scores"][keep],
                              "labels": p["labels"][keep]})
            all_targets.append({"boxes": t["boxes"], "labels": t["labels"]})
    return compute_metrics(all_preds, all_targets, num_classes,
                           iou_thr=iou_thr, class_names=class_names)

def plot_curves(history, out_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epochs, history["train_loss"], "-o", ms=3)
    ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
    ax[0].grid(alpha=0.3)
    ax[1].plot(epochs, history["val_map50"], "-o", ms=3, color="green")
    ax[1].set_title("Val mAP@0.5"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("mAP@0.5")
    ax[1].grid(alpha=0.3)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(); fig.savefig(out_path, dpi=120); plt.close(fig)

def train_pytorch_model(model_wrapper, loaders, device, epochs=50,
                        patience=10, num_classes=3, class_names=None,
                        iou_thr=0.5, score_thr=0.05, logger=None,
                        plots_dir="results/plots"):
    model_wrapper.to(device)
    optimizer, scheduler = model_wrapper.build_optimizer()

    best_map, best_state, no_improve = -1.0, None, 0
    history = {"train_loss": [], "val_map50": []}
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model_wrapper.train_mode()
        running, n = 0.0, 0
        for images, targets in tqdm(loaders["train"], desc=f"epoch {epoch}/{epochs}",
                                    leave=False):
            optimizer.zero_grad()
            loss = model_wrapper.train_forward(images, targets)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_wrapper.parameters(), 10.0)
            optimizer.step()
            running += float(loss.item()); n += 1
        scheduler.step()
        train_loss = running / max(n, 1)

        val = evaluate_model(model_wrapper, loaders["val"], num_classes,
                             iou_thr, score_thr, class_names)
        history["train_loss"].append(train_loss)
        history["val_map50"].append(val["mAP@0.5"])

        msg = (f"[{model_wrapper.name}] epoch {epoch}: loss={train_loss:.4f} "
               f"val mAP@0.5={val['mAP@0.5']:.4f}")
        (logger.info if logger else print)(msg)

        if val["mAP@0.5"] > best_map:
            best_map = val["mAP@0.5"]
            best_state = {k: v.detach().cpu().clone()
                          for k, v in _state_dict(model_wrapper).items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                (logger.info if logger else print)(
                    f"Ранняя остановка на эпохе {epoch} (нет улучшения {patience} эпох)")
                break

    if best_state is not None:
        _load_state_dict(model_wrapper, best_state)

    train_time = time.time() - t0
    plot_curves(history, Path(plots_dir) / f"{model_wrapper.name}_curves.png")

    test = evaluate_model(model_wrapper, loaders["test"], num_classes,
                          iou_thr, score_thr, class_names)
    test["train_time_sec"] = round(train_time, 1)
    return test, history

def _state_dict(wrapper):
    if hasattr(wrapper, "model"):
        return wrapper.model.state_dict()
    return wrapper.bench.state_dict()

def _load_state_dict(wrapper, state):
    if hasattr(wrapper, "model"):
        wrapper.model.load_state_dict(state)
    else:
        wrapper.bench.load_state_dict(state)


In [ ]:
%%writefile scripts/prepare_data.py
from __future__ import annotations

import argparse
import json
import random
import shutil
from collections import Counter
from pathlib import Path

import yaml

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
SPLIT_ALIASES = {"train": "train", "training": "train",
                 "valid": "val", "val": "val", "validation": "val",
                 "test": "test", "testing": "test"}

def imgs_in(d: Path):
    return sorted([p for p in d.rglob("*") if p.suffix.lower() in IMG_EXTS])

def detect_splits(src: Path):
    out = {}
    for child in src.iterdir():
        if child.is_dir() and child.name.lower() in SPLIT_ALIASES:
            out[SPLIT_ALIASES[child.name.lower()]] = child
    return out

def poly_to_bbox(coords):
    xs = coords[0::2]
    ys = coords[1::2]
    return min(xs), min(ys), max(xs), max(ys)

def convert_coco(split_dir: Path, out_dir: Path, name2idx: dict):
    js = list(split_dir.glob("*_annotations.coco.json")) + list(split_dir.glob("*.json"))
    data = json.loads(js[0].read_text())
    images = {im["id"]: im for im in data["images"]}
    cats = {c["id"]: c["name"] for c in data["categories"]}
    anns_by_img = {}
    for a in data["annotations"]:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)
    counter = Counter()

    for img_id, im in images.items():
        w, h = im["width"], im["height"]
        fn = im["file_name"]
        src_img = split_dir / fn
        if not src_img.exists():
            cand = list(split_dir.rglob(Path(fn).name))
            if not cand:
                continue
            src_img = cand[0]
        stem = Path(fn).stem
        shutil.copy2(src_img, out_dir / "images" / src_img.name)
        lines = []
        for a in anns_by_img.get(img_id, []):
            cname = cats.get(a["category_id"], None)
            if cname is None or cname not in name2idx:
                continue
            x, y, bw, bh = a["bbox"]
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            lines.append(f"{name2idx[cname]} {cx:.6f} {cy:.6f} {bw / w:.6f} {bh / h:.6f}")
            counter[cname] += 1
        (out_dir / "labels" / f"{stem}.txt").write_text("\n".join(lines))
    return counter

def convert_yolo(split_dir: Path, out_dir: Path, names: list):
    img_dir = split_dir / "images" if (split_dir / "images").exists() else split_dir
    lab_dir = split_dir / "labels" if (split_dir / "labels").exists() else split_dir
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)
    counter = Counter()
    for img in imgs_in(img_dir):
        shutil.copy2(img, out_dir / "images" / img.name)
        lab = lab_dir / (img.stem + ".txt")
        lines = []
        if lab.exists():
            for line in lab.read_text().strip().splitlines():
                t = line.split()
                if len(t) < 5:
                    continue
                cls = int(float(t[0]))
                vals = list(map(float, t[1:]))
                if len(vals) == 4:
                    cx, cy, bw, bh = vals
                else:
                    x1, y1, x2, y2 = poly_to_bbox(vals)
                    cx, cy, bw, bh = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
                lines.append(f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                counter[names[cls] if cls < len(names) else cls] += 1
        (out_dir / "labels" / f"{img.stem}.txt").write_text("\n".join(lines))
    return counter

def collect_class_names(src: Path, splits: dict):

    for d in splits.values():
        js = list(d.glob("*_annotations.coco.json")) + list(d.glob("*.json"))
        if js:
            data = json.loads(js[0].read_text())

            names = [c["name"] for c in data["categories"]
                     if str(c.get("supercategory", "none")).lower() != "none"]
            if not names:
                names = [c["name"] for c in data["categories"]]
            return sorted(set(names))

    y = src / "data.yaml"
    if y.exists():
        names = yaml.safe_load(y.read_text()).get("names")
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names:
            return sorted(names)
    return None

def is_coco(split_dir: Path):
    return bool(list(split_dir.glob("*_annotations.coco.json")) or list(split_dir.glob("*.json")))

def random_split(src: Path, dst: Path, names, ratios=(0.7, 0.15, 0.15), seed=42):
    img_dir = src / "images" if (src / "images").exists() else src
    images = imgs_in(img_dir)
    random.Random(seed).shuffle(images)
    n = len(images); n_tr = int(n * ratios[0]); n_val = int(n * ratios[1])
    parts = {"train": images[:n_tr], "val": images[n_tr:n_tr + n_val], "test": images[n_tr + n_val:]}
    lab_dir = src / "labels" if (src / "labels").exists() else src
    cnt = Counter()
    for split, files in parts.items():
        (dst / split / "images").mkdir(parents=True, exist_ok=True)
        (dst / split / "labels").mkdir(parents=True, exist_ok=True)
        for img in files:
            shutil.copy2(img, dst / split / "images" / img.name)
            lab = lab_dir / (img.stem + ".txt")
            if lab.exists():
                shutil.copy2(lab, dst / split / "labels" / lab.name)
    return cnt

def eda(dst: Path, names):
    print("\n=== Анализ датасета (EDA) ===")
    total_i = total_b = 0
    for split in ("train", "val", "test"):
        img_dir = dst / split / "images"
        if not img_dir.exists():
            continue
        imgs = imgs_in(img_dir)
        cls = Counter(); nb = 0
        for lab in (dst / split / "labels").glob("*.txt"):
            for line in lab.read_text().strip().splitlines():
                if line.strip():
                    c = int(float(line.split()[0]))
                    cls[names[c] if c < len(names) else c] += 1; nb += 1
        total_i += len(imgs); total_b += nb
        print(f"  {split:5s}: {len(imgs):5d} изобр., {nb:5d} объектов | {dict(cls)}")
    print(f"  ИТОГО: {total_i} изображений, {total_b} объектов")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="распакованный экспорт Roboflow")
    ap.add_argument("--dst", default="data/processed")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    src, dst = Path(args.src), Path(args.dst)
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)

    splits = detect_splits(src)
    names = collect_class_names(src, splits) or \
        ["Glioma_Tumor", "Meningioma_Tumor", "No_Tumor", "Pituitary_Tumor"]
    name2idx = {n: i for i, n in enumerate(names)}
    print(f"Классы ({len(names)}): {names}")

    if splits:
        for canon, d in splits.items():
            if is_coco(d):
                convert_coco(d, dst / canon, name2idx)
            else:
                convert_yolo(d, dst / canon, names)
        if "val" not in splits or "test" not in splits:
            print("ВНИМАНИЕ: отсутствует val или test — проверьте экспорт.")
    else:
        print("Деление на выборки не найдено → случайный сплит 70/15/15")
        random_split(src, dst, names, seed=args.seed)

    data_yaml = {"path": str(dst.resolve()), "train": "train/images",
                 "val": "val/images", "test": "test/images",
                 "nc": len(names), "names": names}
    (dst / "data.yaml").write_text(yaml.safe_dump(data_yaml, allow_unicode=True))
    print(f"\nСоздан {dst / 'data.yaml'}")
    print(f">>> ПРОВЕРЬТЕ: в configs/default.yaml num_classes = {len(names)} и "
          f"class_names = {names}")
    eda(dst, names)

if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

from src.utils.utils import set_seed, get_device, get_logger, load_config, append_metrics

MODEL_KEYS = ["yolov8", "yolov5", "faster_rcnn", "ssd", "efficientdet", "detr"]
YOLO_KEYS = {"yolov8", "yolov5"}

def run_yolo(key, cfg, logger):
    from src.models.yolov8_model import YOLOv8Model
    from src.models.yolov5_model import YOLOv5Model
    seed = cfg["project"]["seed"]
    model = (YOLOv8Model if key == "yolov8" else YOLOv5Model).from_config(cfg, seed=seed)
    data_yaml = cfg["data"]["yaml"]
    logger.info(f"=== Обучение {key} (Ultralytics) ===")
    model.train(data_yaml, epochs=cfg["train"]["epochs"],
                patience=cfg["train"]["early_stop_patience"])
    metrics = model.evaluate(data_yaml, split="test",
                             class_names=cfg["data"]["class_names"])
    return metrics

def run_pytorch(key, cfg, device, logger):
    from src.dataset.dataset import get_dataloaders
    from src.training.train import train_pytorch_model
    from src.models.faster_rcnn import FasterRCNNModel
    from src.models.ssd import SSDModel
    from src.models.efficientdet import EfficientDetModel
    from src.models.detr import DETRModel

    builders = {"faster_rcnn": FasterRCNNModel, "ssd": SSDModel,
                "efficientdet": EfficientDetModel, "detr": DETRModel}
    model = builders[key].from_config(cfg)

    normalize = cfg["normalize"] if model.needs_normalize else None
    loaders = get_dataloaders(
        cfg["data"]["processed_dir"], img_size=model.img_size,
        batch=cfg["models"][key]["batch"], num_workers=cfg["train"]["num_workers"],
        augment=True, aug_cfg=cfg["augment"], normalize=normalize,
    )
    logger.info(f"=== Обучение {key} (PyTorch цикл) ===")
    metrics, _ = train_pytorch_model(
        model, loaders, device, epochs=cfg["train"]["epochs"],
        patience=cfg["train"]["early_stop_patience"],
        num_classes=cfg["data"]["num_classes"],
        class_names=cfg["data"]["class_names"],
        iou_thr=cfg["eval"]["iou_thr"], score_thr=cfg["eval"]["score_thr"],
        logger=logger,
    )
    return metrics

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="yolov8",
                    choices=MODEL_KEYS + ["all"])
    ap.add_argument("--config", default="configs/default.yaml")
    args = ap.parse_args()

    cfg = load_config(args.config)
    set_seed(cfg["project"]["seed"])
    device = get_device(cfg["train"]["device"])
    logger = get_logger("train")
    logger.info(f"Устройство: {device}")

    keys = MODEL_KEYS if args.model == "all" else [args.model]
    for key in keys:
        try:
            if key in YOLO_KEYS:
                metrics = run_yolo(key, cfg, logger)
            else:
                metrics = run_pytorch(key, cfg, device, logger)

            row = {"model": key, **{k: v for k, v in metrics.items()
                                    if k != "per_class"}}
            append_metrics(cfg["eval"]["results_csv"], row)
            logger.info(f"{key} → {json.dumps({k: metrics[k] for k in metrics if k!='per_class'}, ensure_ascii=False)}")

            Path("results/logs").mkdir(parents=True, exist_ok=True)
            with open(f"results/logs/{key}_per_class.json", "w", encoding="utf-8") as f:
                json.dump(metrics.get("per_class", {}), f, ensure_ascii=False, indent=2)
        except Exception as e:
            logger.error(f"Ошибка при обучении {key}: {e}")
            import traceback; logger.error(traceback.format_exc())

    logger.info(f"Готово. Сводка метрик: {cfg['eval']['results_csv']}")

if __name__ == "__main__":
    main()


## 4. Число эпох


In [ ]:
import re
EPOCHS=25
c=open('configs/default.yaml').read(); c=re.sub(r'epochs: \d+',f'epochs: {EPOCHS}',c,count=1); open('configs/default.yaml','w').write(c)


## 5. Датасет (Roboflow)


In [ ]:
from roboflow import Roboflow
rf=Roboflow(api_key="ВАШ_КЛЮЧ")
ds=rf.workspace("mri-brain-tumor-detection").project("brain_tumor_detect").version(1).download("yolov8")
import os; os.environ['DS_DIR']=ds.location


## 6. Подготовка данных


In [ ]:
!python scripts/prepare_data.py --src "$DS_DIR" --dst data/processed


## 7. Обучение


In [ ]:
# YOLOv8n
!python main.py --model yolov8


In [ ]:
# YOLOv5s
!python main.py --model yolov5


In [ ]:
# Faster R-CNN
!python main.py --model faster_rcnn


In [ ]:
# SSD
!python main.py --model ssd


In [ ]:
# EfficientDet-D0
!python main.py --model efficientdet


In [ ]:
# DETR
!python main.py --model detr


## 8. Зависимость качества YOLOv8n от числа эпох


In [ ]:
from ultralytics import YOLO
import pandas as pd, matplotlib.pyplot as plt, os
rows=[]
for ep in [10,25,50]:
    m=YOLO('yolov8n.pt')
    m.train(data='data/processed/data.yaml', epochs=ep, imgsz=640, batch=16, seed=42, project='results/runs', name=f'yolov8_ep{ep}', exist_ok=True, verbose=False)
    b=m.val(data='data/processed/data.yaml', split='test', imgsz=640, project='results/runs', name=f'yolov8_ep{ep}_test', exist_ok=True).box
    rows.append({'epochs':ep,'mAP@0.5':round(float(b.map50),4),'mAP@0.5:0.95':round(float(b.map),4),'precision':round(float(b.mp),4),'recall':round(float(b.mr),4)})
df=pd.DataFrame(rows); df.to_csv('results/logs/ablation_epochs.csv', index=False); print(df.to_string(index=False))
plt.figure(figsize=(7,4)); plt.plot(df['epochs'],df['mAP@0.5'],'-o',label='mAP@0.5'); plt.plot(df['epochs'],df['mAP@0.5:0.95'],'-s',label='mAP@0.5:0.95')
plt.xlabel('Число эпох'); plt.ylabel('Значение метрики'); plt.title('YOLOv8n: зависимость качества от числа эпох'); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig('results/plots/yolov8_epochs_ablation.png', dpi=120); plt.show()


## 9. Результаты + выгрузка


In [ ]:
print(open('results/logs/metrics.csv').read())


In [ ]:
import shutil; from google.colab import files
shutil.make_archive('/content/results_brain_tumor','zip','results')
files.download('/content/results_brain_tumor.zip')
